# Goal 2 (Silver) — Grad Exams from Cosmos · Build it with Copilot

This is a **guided, discovery** notebook. Instead of handing you the code, each step gives you a
**prompt to paste into Fabric's in-notebook Copilot**. Copilot writes the code into the empty cell
below the prompt; you run it and check the **✅ Verify** note. You'll learn *how* to build a Silver
layer — not just copy an answer.

> **How to use Copilot in a Fabric notebook:** open the **Copilot** panel (ribbon), or type your
> request in a cell's Copilot prompt. If the first result isn't quite right, refine the prompt or
> ask Copilot to fix the specific problem ("use `explode_outer` instead", "cast `score` to int", etc.).

## Step 0 — Setup (do this first)

Before you prompt Copilot, prepare the notebook:

1. **Attach the Silver lakehouse.** In the Explorer (left), add / set **`SilverLakehouse`** as the
   **default** lakehouse for this notebook.
2. **Add the Bronze source.** Add the **`Graduate Exams Mirror`** (the Cosmos DB mirror) to this
   notebook as a data source so its `gradExams` table is readable.

You'll write all Silver tables to `SilverLakehouse` under the **`dbo`** schema.

## Step 1 — Attach the Silver lakehouse

**Goal:** Make `SilverLakehouse` the default lakehouse so your tables land in the Silver layer.

**Prompt — copy this into Copilot:**

```text
Add a %%configure cell that sets the default lakehouse for this notebook to the one named SilverLakehouse.
```

**✅ Verify:** A `%%configure` cell targeting **SilverLakehouse** appears; running it restarts the Spark session.

## Step 2 — Read the Bronze Cosmos table

**Goal:** Load the raw graduate-exam documents that Cosmos DB mirrored into OneLake.

**Prompt — copy this into Copilot:**

```text
I've attached the Graduate Exams Mirror to this notebook. It has a Delta table named gradExams in the 'university' schema, mirrored from Cosmos DB. Write PySpark to read that Delta table into a DataFrame named raw, then print its schema and its row count.
```

**✅ Verify:** The schema shows fields like `studentId`, `exam`, `score`, and a nested **`undergradContext`**; the printed row count is greater than 0.

## Step 3 — Normalize the nested `undergradContext`

**Goal:** Cosmos stores a nested object per student. Make it a typed struct so you can reach its fields reliably, whether the mirror stored it as a struct or as a JSON string.

**Prompt — copy this into Copilot:**

```text
The column undergradContext may be stored as a nested struct OR as a JSON string, depending on the mirror. Add a new struct column named uc with this schema: overallGpa double, concentration string, relevantSubjects array<string>, relevantCourseCount long, relevantGpa double, and relevantCourses as an array of struct(courseNumber string, courseName string, letterGrade string, qualityPoints double). If undergradContext is already a struct, use it directly; otherwise parse it with from_json using that schema. Name the resulting DataFrame df.
```

**✅ Verify:** `df.select("uc.*")` runs and returns the typed sub-fields (e.g. `overallGpa`, `relevantCourses`).

## Step 4 — Build Fact 1: `dbo.grad_exam_result`

**Goal:** Flatten one clean row per exam attempt, with snake_case columns and correct types.

**Prompt — copy this into Copilot:**

```text
From df, build a Silver fact DataFrame with these columns (rename and cast): student_id long from studentId, student_name from a trimmed studentName, exam from trimmed exam, target_degree from trimmed targetDegree, attempt_number int from attemptNumber, test_date as a date from testDate, score int, score_scale_min int from scoreScaleMin, score_scale_max int from scoreScaleMax, admit_threshold int from admitThreshold, percentile int, passed boolean, overall_gpa double from uc.overallGpa, concentration from trimmed uc.concentration, relevant_course_count int from uc.relevantCourseCount, and relevant_gpa double from uc.relevantGpa. Drop rows where student_id is null, then de-duplicate on the combination (student_id, exam, attempt_number). Write the result as a managed Delta table named dbo.grad_exam_result using mode overwrite and option overwriteSchema true, and finally print its row count.
```

**✅ Verify:** A managed table **`dbo.grad_exam_result`** exists, columns are snake_case, and a row count prints.

## Step 5 — Build Fact 2: `dbo.grad_exam_relevant_course`

**Goal:** Explode the array of relevant undergrad courses into one row per (student, exam, course).

**Prompt — copy this into Copilot:**

```text
From df, create a second Silver table by exploding the nested array uc.relevantCourses using explode_outer. Produce these columns: student_id long from studentId, exam from trimmed exam, attempt_number int from attemptNumber, course_number from the trimmed array element's courseNumber, course_name from trimmed courseName, letter_grade from trimmed letterGrade, and quality_points double from qualityPoints. Drop rows that have no course, then write the result as a managed Delta table named dbo.grad_exam_relevant_course using mode overwrite and option overwriteSchema true, and print its row count.
```

**✅ Verify:** A managed table **`dbo.grad_exam_relevant_course`** exists; its row count is roughly the number of students times their relevant courses.

## Step 6 — Validate the signal

**Goal:** Confirm the data tells the expected story: stronger relevant-subject GPA tracks with passing.

**Prompt — copy this into Copilot:**

```text
Query dbo.grad_exam_result filtered to attempt_number = 1, grouped by passed, returning: the count of attempts, the average relevant_gpa rounded to 2 decimals, and the average score. Order by passed descending and display the result.
```

**✅ Verify:** The `passed = true` group shows a **higher average relevant GPA** than the `passed = false` group.

## 🎉 Done

You built the **Silver Grad-Exams** tables (`dbo.grad_exam_result`, `dbo.grad_exam_relevant_course`) yourself, using Copilot to translate clear requirements into working
PySpark. These feed the Gold layer in **Goal 4**.

**Next:** try the same discovery approach on your other sources, or move on to the AI enrichment in
**Goal 3**.